## If Snowflake Already Provides Lineage, Why Do Organizations Integrate It with Manta?

In theory, Snowflake's native lineage is enough for many organizations. In practice, companies buy Manta (IBM Manta Data Lineage) when they need lineage **beyond Snowflake**.

---

## What Snowflake Lineage Answers

> "What Snowflake object/column produced this table or column?"

Snowflake lineage tracks:

- Table → Table
- Table → View
- View → View
- Column → Column
- Dynamic Tables
- CTAS, INSERT, MERGE relationships
- Upstream/downstream dependencies **inside Snowflake**

For an organization using only Snowflake, this may be sufficient.

---

## Why Manta Is Still Used

### 1. End-to-End Enterprise Lineage

A typical enterprise data flow looks like:

```
SAP → Informatica → ADF → Snowflake → Power BI
```

Snowflake lineage only knows about the Snowflake portion:

```
STG_CUSTOMER → DIM_CUSTOMER → FACT_SALES
```

It does **not** know:

```
SAP Table → Informatica Mapping → ADF Pipeline → Snowflake Table → Power BI Report
```

Manta can scan all these technologies and build **one end-to-end lineage graph**.

---

### 2. ETL/ELT Tool Lineage

Organizations often use:

- Informatica
- Talend
- DataStage
- ADF
- SSIS
- dbt
- Airflow

The business asks:

> "This metric in Power BI is wrong. Where did it come from?"

Snowflake can show Snowflake dependencies. Manta can show:

```
Power BI Measure
   ↓
Power BI Dataset
   ↓
Snowflake View
   ↓
Snowflake Table
   ↓
ADF Pipeline
   ↓
Oracle Source
```

This is a major reason enterprises invest in Manta.

---

### 3. SQL Parsing Beyond Native Visibility

Suppose you have:

```sql
CREATE VIEW SALES AS
SELECT
  A.CUST_ID,
  B.REGION_NAME,
  SUM(A.AMOUNT) AS TOTAL_SALES
FROM ORDERS A
JOIN REGIONS B;
```

Manta deeply parses SQL code and transformation logic. It can explain:

```
TOTAL_SALES ← SUM(AMOUNT) ← ORDERS.AMOUNT
```

Data governance teams often need this level of transformation tracing for audits.

---

### 4. Legacy Systems

Many enterprises still run:

- Teradata
- Oracle
- Netezza
- SQL Server
- Mainframes

Manta can scan these environments and create lineage across platforms. Snowflake lineage only covers what Snowflake knows about.

---

### 5. Business Impact Analysis

Imagine you want to drop `CUSTOMER_EMAIL`.

Snowflake can show downstream Snowflake objects. Manta may additionally show:

```
CUSTOMER_EMAIL
   ↓
Snowflake View
   ↓
ADF Pipeline
   ↓
Power BI Dashboard
   ↓
Finance Report
```

This helps answer: **"Who will break if I change this column?"**

---

### 6. Regulatory and Audit Requirements

Banks, insurance companies, telecoms, and healthcare organizations often require:

```
Report → Dashboard → Data Warehouse → ETL → Source System
```

Auditors frequently ask:

> "Show the complete lineage of this KPI from source system to report."

Native Snowflake lineage usually covers only part of that journey, whereas Manta is designed specifically for governance and audit use cases.

---

## When Snowflake Lineage Is Enough

You may **not** need Manta if:

- Most data is already in Snowflake
- Transformations are primarily Snowflake SQL or dbt
- Reporting tools are limited
- Governance requirements are moderate

Example architecture where native lineage suffices:

```
Snowflake Landing → Snowflake Staging → Snowflake Mart → Tableau
```

In such architectures, native Snowflake lineage may provide sufficient visibility.

---

## Interview / Architect Answer

If someone asks:

> "Snowflake already has lineage. Why buy Manta?"

A good answer:

> "Snowflake provides excellent object-level and column-level lineage within Snowflake. Manta is typically adopted when organizations need end-to-end lineage across the whole data ecosystem — including ETL tools, source systems, orchestration platforms, BI tools, and legacy databases. Manta helps with impact analysis, governance, audit requirements, and cross-platform visibility that native Snowflake lineage does not fully cover."

That is the distinction most enterprise architects and governance teams care about.

## If Snowflake Has Horizon Catalog, Why Use Alation for Data Catalog?

Snowflake Horizon provides a native catalog within Snowflake. Alation is a standalone, multi-platform data catalog. Organizations adopt Alation when their needs go **beyond what Snowflake alone can serve**.

---

## What Snowflake Horizon Catalog Provides

> "What objects exist in my Snowflake account, and who can access them?"

Horizon covers:

- Object discovery (tables, views, stages, etc.) within Snowflake
- Tags and classification (including auto-classification of PII)
- Object-level and column-level lineage within Snowflake
- Access policies (masking, row access)
- Data quality via DMFs (Data Metric Functions)
- Listing and sharing governance

**Scope limitation:** Horizon only catalogs **Snowflake objects**. It has no visibility into other platforms.

---

## Why Alation Is Still Used

### 1. Multi-Platform Catalog (Single Pane of Glass)

A typical enterprise has data across:

- Snowflake
- Databricks
- Oracle / SQL Server / Postgres
- S3 / ADLS / GCS
- Tableau / Power BI / Looker
- Kafka / Airflow

Alation catalogs **all of these** in one place. Horizon only catalogs Snowflake.

> Business user asks: "Where is customer data?"
>
> - Horizon answers: "In these Snowflake tables."
> - Alation answers: "In Snowflake, Oracle, S3, and this Tableau dashboard."

---

### 2. Business Glossary

Alation provides a rich **business glossary** — a controlled vocabulary mapping business terms to technical objects.

| Concept | Horizon | Alation |
|---|---|---|
| "Revenue" means what? | No formal glossary | Defined term with steward, definition, linked objects |
| "Active Customer" definition | Tag only (no rich context) | Full definition + SQL logic + linked tables + approval workflow |

This matters because different teams often define the same metric differently. Alation enforces a **single source of truth** for business terminology.

---

### 3. Data Search and Discovery (for Non-Technical Users)

Alation is designed for **business analysts and non-technical users**:

- Natural language search across all data sources
- Popularity-based ranking (shows most-used tables first)
- Endorsed/deprecated flags curated by data stewards
- Rich descriptions, articles, and tribal knowledge attached to objects

Horizon's discovery is functional but oriented toward **data engineers** working within Snowsight.

---

### 4. Collaboration and Tribal Knowledge

Alation captures institutional knowledge:

- **Conversations:** Q&A threads attached to tables/columns
- **Articles:** Wiki-like documentation pages
- **Queries:** Saved and shared SQL with usage context
- **Endorsements/Warnings:** Stewards can flag "use this" or "do not use this"

Horizon has tags and comments, but lacks this collaborative layer.

---

### 5. Query Log Analysis and Usage Intelligence

Alation ingests **query logs** from Snowflake (and other platforms) to determine:

- Which tables are most queried (popularity)
- Which columns are actually used in JOINs and filters
- Who are the top users of a dataset
- Which tables are stale or unused

This intelligence drives better search results and helps governance teams identify unused objects for cleanup.

---

### 6. Governance Workflows and Stewardship

Alation provides:

```
Data Steward assigns ownership
   ↓
Business term proposed
   ↓
Review and approval workflow
   ↓
Term published to glossary
   ↓
Linked to physical objects
```

Horizon has role-based access control and tagging, but does **not** have built-in approval workflows, stewardship assignment, or governance process automation.

---

### 7. BI Tool Cataloging

Alation catalogs BI layers:

- Tableau workbooks, dashboards, calculated fields
- Power BI datasets, measures, reports
- Looker explores, views, dimensions

This lets users trace from a **dashboard metric** back to the **source table** — all within Alation.

Horizon does not catalog BI tools.

---

## When Snowflake Horizon Is Enough

You may **not** need Alation if:

- Snowflake is your only (or primary) data platform
- Users are mostly data engineers comfortable with Snowsight
- Governance needs are met by tags, masking policies, and RBAC
- No formal business glossary is required
- BI tooling is minimal or single-vendor

Example architecture where Horizon suffices:

```
Snowflake (all data) → dbt (transforms) → one BI tool
```

---

## Side-by-Side Comparison

| Capability | Snowflake Horizon | Alation |
|---|---|---|
| Catalog Snowflake objects | ✅ | ✅ |
| Catalog non-Snowflake sources | ❌ | ✅✅ Strong |
| Business glossary | ❌ | ✅✅ Core strength |
| Data lineage (within Snowflake) | ✅ | ✅ (via ingestion) |
| Cross-platform lineage | ❌ | ✅ Moderate |
| Natural language search | ⚠️ Basic | ✅✅ Strong |
| Popularity / usage intelligence | ⚠️ Basic | ✅✅ Strong |
| Collaboration (Q&A, articles) | ❌ | ✅✅ Core strength |
| Governance workflows | ⚠️ Basic | ✅✅ Strong |
| BI tool cataloging | ❌ | ✅✅ Strong |
| Access control / masking | ✅✅ Core strength | ❌ (delegates to source) |
| Data quality (DMFs) | ✅ | ⚠️ Basic |
| PII classification | ✅ | ✅ |

---

## Interview / Architect Answer

If someone asks:

> "Snowflake has Horizon. Why buy Alation?"

A good answer:

> "Snowflake Horizon is an excellent native catalog for governing Snowflake objects — access control, classification, lineage, and quality within Snowflake. Alation is adopted when organizations need a unified catalog across multiple data platforms, a business glossary for consistent terminology, collaboration features for tribal knowledge, BI tool cataloging, and governance workflows with stewardship. Horizon governs Snowflake; Alation governs the enterprise data ecosystem."

---

## Key Distinction

```
Horizon  = Govern what is INSIDE Snowflake (deep, native, policy-enforcing)
Alation  = Discover and understand data ACROSS the enterprise (broad, collaborative, knowledge-sharing)
```

They are **complementary**, not competing. Many organizations use both: Horizon for enforcement, Alation for discovery and collaboration.

## If Snowflake Has Data Governance Features, Why Use Collibra DQ?

Snowflake provides native data quality via DMFs (Data Metric Functions) and governance via Horizon. Collibra DQ (formerly Owl DQ) is a dedicated **data quality platform**. Organizations adopt Collibra DQ when they need data quality capabilities **far beyond what Snowflake natively offers**.

---

## What Snowflake Provides for Data Quality

> "Are my Snowflake tables meeting basic quality expectations?"

Snowflake's native data quality covers:

- **DMFs (Data Metric Functions):** User-defined or system-provided functions that measure quality metrics on tables/columns
- **System DMFs:** NULL_COUNT, DUPLICATE_COUNT, UNIQUE_COUNT, FRESHNESS
- **Scheduling:** DMFs can run on a schedule via Data Quality Monitoring
- **Alerts:** Trigger notifications when thresholds are breached
- **Tags and classification:** Identify sensitive data

**Scope limitation:** Snowflake DMFs operate **only on Snowflake tables**, provide basic metric types, and lack advanced profiling, ML-driven anomaly detection, and cross-platform quality scoring.

---

## Why Collibra DQ Is Still Used

### 1. Multi-Platform Data Quality

A typical enterprise has data across:

- Snowflake
- Databricks / Spark
- Oracle / SQL Server / Postgres
- S3 / ADLS (files: CSV, Parquet, JSON)
- Kafka streams
- APIs and SaaS applications

Collibra DQ monitors quality **across all of these** with one consistent framework.

> Data team asks: "Is our customer data consistent across all systems?"
>
> - Snowflake DMFs answer: "Quality is fine in Snowflake tables."
> - Collibra DQ answers: "Snowflake is fine, but Oracle source has 12% duplicates and S3 files have schema drift."

---

### 2. Automated Data Profiling

Collibra DQ automatically profiles datasets **without writing rules**:

| Capability | Snowflake DMFs | Collibra DQ |
|---|---|---|
| Null count | ✅ (system DMF) | ✅ |
| Duplicate count | ✅ (system DMF) | ✅ |
| Statistical profiling (mean, stddev, distribution) | ❌ | ✅ Automatic |
| Pattern detection (email, phone, SSN formats) | ❌ | ✅ Automatic |
| Cardinality analysis | ❌ | ✅ Automatic |
| Outlier detection | ❌ | ✅ ML-driven |
| Schema drift detection | ❌ | ✅ Automatic |

Collibra DQ learns the **expected shape** of your data and flags deviations — no manual rule writing required.

---

### 3. ML-Based Anomaly Detection (Self-Learning Rules)

Snowflake DMFs require you to **define thresholds manually**:

```sql
-- Snowflake: you must know what "bad" looks like
SELECT NULL_COUNT(COLUMN) FROM TABLE;
-- Then manually set: IF null_count > 100, alert
```

Collibra DQ uses **machine learning** to:

- Learn normal data behavior over time
- Automatically detect anomalies without hardcoded thresholds
- Adapt as data patterns evolve

```
Week 1-4: Collibra learns "orders table gets 10K-15K rows daily"
Week 5:   Only 2K rows arrive → Automatic anomaly flagged
```

No manual rule needed. Snowflake DMFs cannot do this.

---

### 4. Data Quality Scoring (DQ Score)

Collibra DQ assigns a **composite quality score** to every dataset:

```
CUSTOMER table:
  Completeness:  94%
  Accuracy:      87%
  Uniqueness:    99%
  Timeliness:    92%
  Consistency:   88%
  ─────────────────
  Overall DQ Score: 92/100
```

Snowflake DMFs return individual metric values but do **not** compute a composite score, trend it over time, or compare it across datasets.

---

### 5. Data Observability (Pipeline-Level Monitoring)

Collibra DQ monitors the **entire data pipeline**, not just table contents:

```
Source (Oracle)
   ↓  ← Row count check
ETL (ADF)
   ↓  ← Schema validation
Landing (Snowflake RAW)
   ↓  ← Freshness check
Transformed (Snowflake MART)
   ↓  ← Accuracy rules
BI (Power BI)
   ↓  ← Reconciliation
```

It detects:

- **Volume anomalies:** "Expected 1M rows, got 100K"
- **Freshness issues:** "Table not refreshed in 6 hours"
- **Schema changes:** "Column REGION was dropped upstream"
- **Reconciliation failures:** "Source has 1M rows, target has 980K"

Snowflake DMFs only check what is **already in Snowflake** — they have no visibility into upstream pipelines.

---

### 6. Cross-System Reconciliation

Collibra DQ can compare data **between platforms**:

```
Oracle (source): 1,000,000 customers
     vs.
Snowflake (target): 998,500 customers
     → 1,500 records missing → Alert
```

Snowflake DMFs operate on a single table. They cannot compare across systems.

---

### 7. Business Rule Engine (Complex Rules)

Collibra DQ supports complex, multi-table, multi-condition business rules:

```
Rule: "Every ORDER must have a valid CUSTOMER_ID in the CUSTOMER table"
Rule: "TOTAL_AMOUNT must equal SUM(LINE_ITEMS) for each order"
Rule: "No customer under 18 should have a CREDIT_CARD product"
```

Snowflake DMFs are **column-level functions** — they return a single metric per column. Multi-table referential integrity checks or complex business logic rules require custom SQL procedures, not DMFs.

---

### 8. Historical Trending and Root Cause Analysis

Collibra DQ stores quality metrics over time and provides:

- Trend dashboards: "Quality score was 98% last month, now 85% — what changed?"
- Root cause drill-down: "Drop caused by NULL values in REGION column since March 12"
- SLA tracking: "This table must maintain 95% completeness for regulatory compliance"

Snowflake DMF results can be stored in a table manually, but there is no built-in trending, root cause analysis, or SLA framework.

---

## When Snowflake DMFs Are Enough

You may **not** need Collibra DQ if:

- All data lives in Snowflake
- Quality checks are simple (nulls, duplicates, freshness)
- You can define thresholds manually (no need for ML-based detection)
- No cross-system reconciliation is required
- Governance/audit does not require composite DQ scores or historical trends
- Team is comfortable writing custom DMFs for complex rules

Example architecture where native quality suffices:

```
Snowflake (source of truth) → dbt (transforms + tests) → DMFs (monitoring) → Alerts
```

---

## Side-by-Side Comparison

| Capability | Snowflake (DMFs + Horizon) | Collibra DQ |
|---|---|---|
| Basic null/duplicate checks | ✅ | ✅ |
| Freshness monitoring | ✅ | ✅ |
| Multi-platform monitoring | ❌ Snowflake only | ✅✅ Core strength |
| Automated profiling (no rules) | ❌ | ✅✅ Core strength |
| ML-based anomaly detection | ❌ | ✅✅ Core strength |
| Composite DQ scoring | ❌ | ✅✅ Core strength |
| Pipeline observability | ❌ | ✅✅ Strong |
| Cross-system reconciliation | ❌ | ✅✅ Strong |
| Complex business rules | ⚠️ Custom SQL needed | ✅ Built-in rule engine |
| Historical trending / SLA | ⚠️ Manual | ✅✅ Strong |
| Root cause analysis | ❌ | ✅ |
| Schema drift detection | ❌ | ✅ |
| Access control / masking | ✅✅ Core strength | ❌ (not its role) |
| Data catalog / discovery | ✅ | ❌ (not its role) |

---

## Interview / Architect Answer

If someone asks:

> "Snowflake has DMFs and Horizon. Why buy Collibra DQ?"

A good answer:

> "Snowflake DMFs provide solid native quality checks — nulls, duplicates, freshness — for data already in Snowflake. Collibra DQ is adopted when organizations need automated profiling without manual rules, ML-driven anomaly detection, composite quality scoring, cross-system reconciliation, pipeline-level observability, and historical trending with SLA tracking. Snowflake monitors table health; Collibra DQ monitors enterprise data health across the entire ecosystem."

---

## Key Distinction

```
Snowflake DMFs = Check specific metrics on Snowflake tables (reactive, rule-based, single-platform)
Collibra DQ    = Observe, score, and alert on data quality across all systems (proactive, ML-driven, multi-platform)
```

They are **complementary**. Many organizations run Snowflake DMFs for lightweight in-platform checks and Collibra DQ for enterprise-grade quality governance and observability.

## If Snowflake Has Tasks and Task Graphs (DAGs), Why Use Apache Airflow?

> **Note:** Your question is valid. Snowflake Tasks are indeed the native equivalent of a workflow orchestrator within Snowflake. Airflow is the external counterpart. The comparison is:
> - **Snowflake Tasks** = Orchestrate work **inside Snowflake**
> - **Apache Airflow** = Orchestrate work **across the entire data ecosystem**

Organizations adopt Airflow when their workflows span **beyond Snowflake** or require capabilities Snowflake Tasks do not provide.

---

## What Snowflake Tasks Provide

> "Run SQL/procedures on a schedule or in a dependency chain within Snowflake."

Snowflake Tasks cover:

- **Scheduled execution:** CRON or interval-based scheduling
- **Task graphs (DAGs):** Chain tasks with parent-child dependencies
- **Stream-driven triggers:** Run when new data arrives (SYSTEM$STREAM_HAS_DATA)
- **Serverless or warehouse-based compute**
- **Error handling:** SUSPEND_TASK_AFTER_NUM_FAILURES, finalizer tasks
- **Return values:** Pass data between tasks via SYSTEM$SET_RETURN_VALUE

**Scope limitation:** Snowflake Tasks can **only execute Snowflake SQL, procedures, and Snowpark functions**. They cannot orchestrate external systems.

---

## Why Airflow Is Still Used

### 1. Cross-Platform Orchestration

A typical enterprise data pipeline:

```
Extract from Salesforce API
   ↓
Land files in S3
   ↓
Trigger Spark job on Databricks
   ↓
Load into Snowflake
   ↓
Run dbt transformations
   ↓
Refresh Tableau extract
   ↓
Send Slack notification
```

Snowflake Tasks can only handle the Snowflake portion. Airflow orchestrates **the entire pipeline end-to-end**.

---

### 2. Extensive Connector Ecosystem (Operators/Hooks)

Airflow has **2,000+ pre-built operators** for:

| Category | Examples |
|---|---|
| Cloud storage | S3, ADLS, GCS |
| Databases | Snowflake, Postgres, Oracle, MySQL, BigQuery |
| APIs | Salesforce, HubSpot, Stripe, REST/HTTP |
| Compute | Databricks, EMR, Spark, Kubernetes |
| Messaging | Kafka, SQS, Pub/Sub |
| BI tools | Tableau, Power BI, dbt Cloud |
| Notifications | Slack, Email, PagerDuty, Teams |

Snowflake Tasks have **zero external connectors**. They only execute Snowflake-native operations.

---

### 3. Complex Dependency Logic

Airflow supports:

```python
# Branching: take different paths based on conditions
with DAG('my_pipeline'):
    check = BranchPythonOperator(task_id='check_data_quality')
    path_a = SnowflakeOperator(task_id='full_refresh')
    path_b = SnowflakeOperator(task_id='incremental_load')
    check >> [path_a, path_b]  # conditional branching
```

- **Branching:** Execute different paths based on runtime conditions
- **Dynamic DAGs:** Generate tasks programmatically based on config/metadata
- **Cross-DAG dependencies (sensors):** Wait for another pipeline to finish
- **Trigger rules:** Run only if all parents succeed, or if at least one succeeds, etc.

Snowflake Task graphs support linear/fan-out dependencies and finalizer tasks, but **no conditional branching, dynamic generation, or cross-graph sensors**.

---

### 4. Non-SQL Workloads

Airflow can orchestrate:

- Python scripts (data science, ML training)
- Shell/bash commands
- Docker containers
- Kubernetes pods
- Spark/Databricks jobs
- API calls (REST, GraphQL)
- File transfers (SFTP, FTP)

Snowflake Tasks execute **only SQL, stored procedures, or Snowpark (Python/Java/Scala within Snowflake)**. They cannot run arbitrary external code.

---

### 5. Centralized Monitoring and Alerting

Airflow provides:

- **Web UI:** Visual DAG view, task status, Gantt charts, logs
- **SLA monitoring:** Alert if a task misses its deadline
- **Custom alerting:** Email, Slack, PagerDuty on failure/retry/SLA miss
- **Audit logs:** Full execution history with parameters and duration
- **Unified view:** All pipelines (Snowflake + non-Snowflake) in one dashboard

Snowflake task monitoring:

- TASK_HISTORY() function
- TASK_DEPENDENTS() for graph view
- Alerts (separate object)
- Snowsight UI (Snowflake tasks only)

When you have 50 pipelines across 5 platforms, Airflow provides **one place** to monitor everything.

---

### 6. Version Control and CI/CD (DAGs as Code)

Airflow DAGs are **Python files** stored in Git:

```
repo/
├── dags/
│   ├── customer_pipeline.py
│   ├── sales_pipeline.py
│   └── ml_training.py
├── tests/
└── .github/workflows/deploy.yml
```

Benefits:

- Pull requests and code review for pipeline changes
- Unit testing of DAG logic
- CI/CD deployment (promote dev → staging → prod)
- Full Git history of every pipeline change

Snowflake Tasks are **DDL objects** (CREATE TASK). They can be version-controlled via scripts, but there is no native Git integration, PR workflow, or testing framework for task definitions.

---

### 7. Backfill and Replay

Airflow excels at historical reprocessing:

```bash
# Re-run pipeline for all of March 2024
airflow dags backfill --start-date 2024-03-01 --end-date 2024-03-31 my_pipeline
```

- Automatically partitions work by execution_date
- Replays failed runs with one click
- Catches up after outages

Snowflake Tasks have no native backfill mechanism. You must manually re-execute or write custom logic.

---

### 8. Multi-Tenant / Multi-Environment Support

Airflow supports:

```
Dev environment  → dev Snowflake + dev S3 + dev Databricks
Staging          → staging Snowflake + staging S3
Production       → prod Snowflake + prod S3 + prod Databricks
```

One DAG definition, parameterized per environment. Snowflake Tasks exist in **one account at a time** — multi-environment promotion requires external tooling (Terraform, Schemachange, etc.).

---

## When Snowflake Tasks Are Enough

You may **not** need Airflow if:

- All orchestration is within Snowflake (SQL transforms, procedures)
- Dependencies are simple (linear or fan-out, no conditional branching)
- No external systems need triggering (APIs, Spark, file transfers)
- Stream-triggered incremental processing covers your needs
- Team prefers SQL-based definitions over Python code

Example architecture where Snowflake Tasks suffice:

```
Streams (CDC) → Task Graph (transform chain) → Final tables → Snowflake Alert on failure
```

---

## Side-by-Side Comparison

| Capability | Snowflake Tasks | Apache Airflow |
|---|---|---|
| Schedule SQL/procedures | ✅ | ✅ |
| DAG (dependency chains) | ✅ | ✅ |
| Stream-triggered execution | ✅✅ Core strength | ⚠️ Requires sensor/polling |
| Serverless compute | ✅ | ❌ (needs infra) |
| Cross-platform orchestration | ❌ | ✅✅ Core strength |
| Conditional branching | ❌ | ✅✅ Core strength |
| Dynamic DAG generation | ❌ | ✅✅ Strong |
| 2,000+ connectors | ❌ | ✅✅ Core strength |
| Non-SQL workloads | ❌ | ✅✅ Core strength |
| Backfill / replay | ❌ | ✅✅ Core strength |
| Git-based CI/CD | ❌ | ✅✅ Strong |
| Visual monitoring (all pipelines) | ⚠️ Snowflake only | ✅✅ Strong |
| SLA monitoring | ⚠️ Basic (alerts) | ✅✅ Strong |
| Multi-environment promotion | ⚠️ Manual | ✅ Built-in |
| Zero infrastructure needed | ✅✅ Core strength | ❌ (self-hosted or managed) |
| Simplicity (no Python needed) | ✅✅ Core strength | ❌ (Python required) |

---

## Interview / Architect Answer

If someone asks:

> "Snowflake has Tasks and DAGs. Why use Airflow?"

A good answer:

> "Snowflake Tasks are excellent for orchestrating SQL-based workflows entirely within Snowflake — scheduled transforms, stream-triggered processing, and simple dependency chains with zero infrastructure. Airflow is adopted when organizations need cross-platform orchestration across APIs, databases, cloud services, and compute engines; conditional branching and dynamic DAG logic; backfill capabilities; Git-based CI/CD; and centralized monitoring of all pipelines regardless of platform. Snowflake Tasks orchestrate Snowflake; Airflow orchestrates the enterprise data ecosystem."

---

## Key Distinction

```
Snowflake Tasks = Run Snowflake work on a schedule with dependencies (simple, native, zero-infra)
Apache Airflow  = Orchestrate any work anywhere with complex logic (powerful, multi-platform, code-first)
```

They are **complementary**. A common pattern: Airflow triggers a Snowflake Task graph as one step in a larger cross-platform pipeline.

## Conclusion

Snowflake provides a robust set of native capabilities — governance (Horizon), lineage, data quality (DMFs), orchestration (Tasks), and cataloging — that satisfy the requirements of most organizations operating primarily within the Snowflake ecosystem.

However, enterprises rarely operate on a single platform. When data flows through multiple systems — source databases, ETL tools, cloud storage, BI platforms, and legacy infrastructure — the need arises for specialized tools that extend visibility and control **beyond Snowflake's boundaries**.

| Need | Snowflake Native | External Tool |
|---|---|---|
| Lineage across all platforms | ❌ | Manta |
| Enterprise data catalog + glossary | ⚠️ Basic | Alation |
| ML-driven data quality at scale | ⚠️ Basic | Collibra DQ |
| Cross-platform orchestration | ❌ | Airflow |

**The decision rule is straightforward:**

- If your data lives **mostly in Snowflake** and governance needs are moderate → Snowflake native features are sufficient.
- If your data spans **multiple platforms** and you need enterprise-grade lineage, cataloging, quality, or orchestration → integrate specialized tools with Snowflake.

These tools do not replace Snowflake — they **complement** it by covering what Snowflake was never designed to handle: the world outside Snowflake.

---

## Comparison Chart

| Capability              | Snowflake Horizon | Alation           | Collibra DQ      | Manta (IBM)         |
| ----------------------- | ----------------- | ----------------- | ---------------- | ------------------- |
| Data Catalog            | ✅                 | ✅✅ Strong         | ⚠️ Limited       | ⚠️ Limited          |
| Business Glossary       | ✅                 | ✅✅ Strong         | ❌                | ❌                   |
| Data Lineage            | ✅                 | ✅                 | ❌ (not primary)  | ✅✅ Core strength    |
| Data Discovery          | ✅                 | ✅✅ Strong         | ❌                | ⚠️ Limited          |
| Governance Workflows    | ✅ Basic/Moderate  | ✅ Moderate/Strong | ⚠️ Limited       | ⚠️ Limited          |
| Data Quality Monitoring | ⚠️ Basic          | ⚠️ Basic          | ✅✅ Core strength | ❌                   |
| Data Quality Rules      | ⚠️                | ⚠️                | ✅                | ❌                   |
| Data Profiling          | ⚠️                | ⚠️                | ✅                | ❌                   |
| Data Observability      | ❌                 | ❌                 | ✅                | ⚠️ Limited          |
| SQL Parsing / Code Lineage | ⚠️ Basic       | ⚠️ Basic          | ❌                | ✅✅ Core strength    |
| Cross-Platform Lineage  | ❌                 | ✅ Moderate        | ❌                | ✅✅ Core strength    |
| Legacy System Support   | ❌                 | ⚠️ Limited        | ❌                | ✅✅ Strong           |

## Databricks Orchestration vs Airflow

### The Core Question

> Why do people use Airflow with Databricks? Databricks notebooks require PySpark and a Spark environment. Does Airflow run Databricks notebooks outside Databricks? Who provides the compute power?

---

## The Answer (Most Important Concept)

**Airflow does NOT run Databricks notebooks. Databricks runs Databricks notebooks. Airflow only tells Databricks WHEN to run them.**

```
Airflow  = Project Manager (coordinates, monitors, decides)
Databricks = Worker (executes, computes, processes)
```

---

## What Happens When Airflow Triggers Databricks?

```
Airflow Server
   │
   │  REST API call: "Run notebook sales_transformation.py"
   ▼
Databricks receives the request
   │
   ▼
Databricks Cluster starts (Azure VMs / AWS EC2 / GCP Compute)
   │
   ▼
Notebook executes (Spark code runs HERE, not in Airflow)
   │
   ▼
Cluster stops → Airflow receives "success" status
```

**Compute comes from Databricks, NOT from Airflow.**

---

## Who Provides Compute in Each Scenario?

| Scenario | Airflow Does | Compute Provider |
|---|---|---|
| Airflow → Databricks Notebook | Sends "Run Notebook XYZ" via API | Databricks Cluster (VMs underneath) |
| Airflow → Snowflake SQL | Sends `CALL RUN_DAILY_LOAD()` | Snowflake Warehouse |
| Airflow → Power BI Refresh | Sends refresh API call | Power BI Service |

Airflow **never** executes the heavy computation. It delegates to the platform that owns the compute.

---

## Does Airflow Execute Spark Code?

**No.** This is where beginners get confused.

Airflow does this:

```python
# Airflow task (runs in seconds — just an API call)
trigger_databricks_job(notebook_path="/sales_transformation")
```

Airflow does **NOT** do this:

```python
# This runs INSIDE Databricks, not Airflow
spark.read.parquet("s3://data/sales").groupBy("region").sum("amount")
```

The Airflow task typically lasts only a few seconds to submit the job, then polls for completion.

---

## Why Use Airflow Then?

Because enterprise workflows span multiple platforms:

```
Receive File (S3/ADLS)
   ↓
Run Databricks Notebook (Spark compute)
   ↓
Load into Snowflake (SQL compute)
   ↓
Refresh Power BI (BI service)
   ↓
Send Email notification
```

- Databricks Workflows only know how to orchestrate **Databricks jobs**.
- Snowflake Tasks only know how to orchestrate **Snowflake jobs**.
- Airflow orchestrates **everything together**.

---

## Real Enterprise Example

```
Airflow DAG: daily_sales_pipeline
├── Task 1: Wait for file on S3           (S3 Sensor)
├── Task 2: Run Databricks notebook       (Databricks Operator → Spark compute)
├── Task 3: Run Snowflake procedure       (Snowflake Operator → Warehouse compute)
├── Task 4: Refresh Power BI dataset      (HTTP Operator → Power BI service)
└── Task 5: Send Slack notification       (Slack Operator)
```

Airflow provides: dependency management, monitoring, retry logic, SLA tracking, and alerting.

Each platform provides: **its own compute**.

---

## Analogy

| Food Delivery | Data Pipeline |
|---|---|
| Swiggy/Zomato (app) | Airflow |
| Restaurant (kitchen) | Databricks / Snowflake |
| Cook (prepares food) | Spark Cluster / Warehouse |

Swiggy does **not** cook the food. It coordinates order placement, status monitoring, and delivery tracking.

Airflow does **not** run Spark. It coordinates Databricks.

---

## Then Why Not Just Use Databricks Workflows?

**If your pipeline is Databricks-only:**

```
Notebook A → Notebook B → Notebook C  (all in Databricks)
```

✅ **Databricks Workflows is enough.** No need for Airflow.

**If your pipeline spans multiple technologies:**

```
API → Databricks → Snowflake → Salesforce → Power BI → Email
```

✅ **Airflow becomes valuable** because it orchestrates across all of them.

---

## The Architecture Most Companies Use

| Team Type | Orchestration | Pattern |
|---|---|---|
| Databricks-only team | Databricks Workflows | Workflows → Notebooks |
| Enterprise Data Platform | Airflow | Airflow → Databricks + Snowflake + BI + APIs |

```
Enterprise Architecture:

┌─────────────────────────────────────────┐
│              AIRFLOW (Orchestrator)      │
├─────────┬──────────┬──────────┬─────────┤
│         ▼          ▼          ▼         │
│   Databricks   Snowflake   Power BI     │
│   (Spark)      (SQL)       (Reports)    │
│                                         │
│   Each provides its OWN compute         │
└─────────────────────────────────────────┘
```

---

## Summary

| Question | Answer |
|---|---|
| Does Airflow run Spark code? | No. Databricks does. |
| Does Airflow provide compute? | No. It only sends API calls. |
| Who provides compute for notebooks? | Databricks Cluster (VMs). |
| Is Databricks just storing notebooks? | No. It provides the Spark engine, storage integration, and compute. |
| When is Airflow needed? | When workflows span multiple platforms. |
| When is Databricks Workflows enough? | When everything is inside Databricks. |